# Custom Optical Water Types in the Loire estuary plume

The built-in OWT databases (Spyrakos et al. 2018, Bi and Hieronymi 2024) describe
waters sampled worldwide. Regional waters, such as a river plume with a sharp gradient
of suspended sediments, may be poorly represented by them. GRSl2bgen lets you
classify an image with **any set of reference spectra**. This tutorial adapts the
{doc}`../grsl2bgen_owt_manual_selection` tutorial to a turbid coastal area and shows
how to:

1. select reference spectra directly from the image (manual selection);
2. classify the image with these spectra and check the result;
3. compare with the built-in databases;
4. build a combined database from external spectra, as done for the Tarasenko et al.
   (2025) database shipped with GRSl2bgen (Bi2024 + Gernez et al. 2023 phytoplankton
   types);
5. use simulated spectra of harmful algal species, and tune the spectral settings.

The example is a Sentinel-2A image of the **Loire estuary** (France) acquired on
12 April 2017: the turbid plume of the Loire spreads over the continental shelf of
the Bay of Biscay, between the clear offshore waters and the shallow Bay of
Bourgneuf to the south. See {doc}`../grsl2bgen_owt_minimal` for an introduction to
the OWT classification and {doc}`../../algorithms` for the equations.

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # keep the notebook output readable

import glob
import logging
import os

import matplotlib as mpl
import matplotlib.patheffects  # noqa: F401, outlined labels
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray  # noqa: F401, registers the .rio accessor
from rasterio.warp import transform

import GRSl2bgen

logging.getLogger().setLevel(logging.WARNING)

# lighter figures for the documentation
%config InlineBackend.figure_formats = ['jpeg']
plt.rcParams['figure.dpi'] = 72

print(f'GRSl2bgen {GRSl2bgen.__version__}')

## Load the L2A image

The GRS L2A product is a subset of tile 30TWT stored as a **multiscale Zarr**
store: groups `0`, `1` and `2` hold the image at full (20 m), half and quarter
resolution, and the group `ancillary` holds the atmospheric data. Open the
full-resolution group with `xr.open_zarr` and pass the dataset to
{py:class}`GRSl2bgen.Product`. The subset (35 x 35 km) is used as is.

In [ ]:
l2a_path = ('/data/tmp/grstbx_subset/loire_estuary/'
            'S2A_MSIL2AGRS_20170412T110621_N0500_R137_T30TWT_20231111T110836_V3.0.1dev_loire_estuary.zarr')

# group '0' = full resolution of the multiscale store
prod = GRSl2bgen.Product(xr.open_zarr(l2a_path, group='0', decode_coords='all'))
raster = prod.raster

# keep the valid water pixels only (mask == 0)
Rrs = raster.Rrs.where(raster.mask == 0).load()
date = str(raster.time.dt.strftime('%Y-%m-%d').values)
dx, dy = (abs(r) for r in raster.rio.resolution())
print(f'{date}: {raster.sizes["y"]} x {raster.sizes["x"]} pixels of {dx:.0f} x {dy:.0f} m, '
      f'bands {Rrs.wl.values.tolist()} nm')

True-colour composite (left) and $R_{rs}(665)$ (right). In the red, the reflectance
increases with the concentration of suspended particulate matter (SPM): the plume
leaves the estuary mouth (east) and is driven offshore by the ebb and the river
discharge, with fronts and filaments over the shelf.

In [ ]:
def show_map(ax, title):
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_axis_off()

rgb = Rrs.sel(wl=[665, 560, 490])
fig, axs = plt.subplots(1, 2, figsize=(15, 6.5), constrained_layout=True)
rgb.plot.imshow(rgb='wl', robust=True, ax=axs[0])
show_map(axs[0], f'True colour, {date}')
Rrs.sel(wl=665).plot.imshow(vmin=0, vmax=0.02, cmap='viridis', ax=axs[1],
                            cbar_kwargs={'label': r'$R_{rs}(665)$ (sr$^{-1}$)', 'shrink': 0.8})
show_map(axs[1], 'Red, 665 nm')

## Select reference spectra from the image

Each reference spectrum is the median $R_{rs}$ of a 3 x 3 pixel window around a point
chosen for a distinct water type. The points are given here by their map coordinates
(m, UTM) so that the notebook is reproducible.

```{tip}
In Jupyter, you can pick the points interactively with the `ViewSpectral` tool of
[grstbx](https://github.com/Tristanovsk/grstbx): click on the map to add points, then
retrieve their coordinates.

    import grstbx
    v = grstbx.visual.ViewSpectral(Rrs, reproject=True)
    v.visu()                      # click on the map
    points = v.poi_stream.data    # x, y of the clicked points (EPSG:3857)

Convert them to the image projection before extracting the spectra.
```

In [ ]:
points = {
    'offshore':      (523000, 5230000),
    'dark filament': (530500, 5232500),
    'shelf':         (535000, 5215000),
    'south shelf':   (540000, 5207000),
    'plume edge':    (540000, 5230000),
    'plume':         (546000, 5228000),
    'plume core':    (553000, 5228500),
    'Bourgneuf bay': (552000, 5203500),
    'north coast':   (549000, 5234500),
}

half = 1.5 * dx   # half-width of the 3 x 3 window
spectra = []
for name, (x, y) in points.items():
    window = Rrs.sel(x=slice(x - half, x + half), y=slice(y + half, y - half))
    spectra.append(window.median(['x', 'y']).assign_coords(owt=name))

xowt_img = xr.concat(spectra, dim='owt').reset_coords(drop=True).transpose('owt', 'wl')
xowt_img.name = 'Rrs'
names = xowt_img.owt.values
Nowt = len(names)
colors = plt.get_cmap('tab10')(np.arange(Nowt))
cmap_img = mpl.colors.ListedColormap(colors)
xowt_img

Position of the points (left) and their spectra (right). The spectra follow the
sediment gradient: from the clear offshore and shelf waters, with a maximum at
490--560 nm and a low red and NIR signal, to the plume, where the maximum shifts to
560 nm and the red and NIR reflectance increases with the SPM concentration. In the
plume core, the reflectance stays high up to 705 nm and shows a NIR plateau
(783--842 nm) typical of highly turbid waters. The "dark filament" in the north-west
has a lower blue-green reflectance and a relatively higher red reflectance than the
offshore waters.

In [ ]:
outline = [mpl.patheffects.withStroke(linewidth=3, foreground='black')]
fig, axs = plt.subplots(1, 2, figsize=(17, 6.5), constrained_layout=True)
rgb.plot.imshow(rgb='wl', robust=True, ax=axs[0])
for i, (name, (x, y)) in enumerate(points.items()):
    axs[0].scatter(x, y, color=colors[i], edgecolor='white', s=90, zorder=10)
    right = x > 548000   # points close to the eastern edge: label on their left
    axs[0].annotate(name, (x, y), xytext=(-8 if right else 8, -12 if right else 4),
                    textcoords='offset points', ha='right' if right else 'left',
                    color='white', fontsize=11, path_effects=outline)
show_map(axs[0], 'Selected points')

for i, name in enumerate(names):
    xowt_img.sel(owt=name).plot(ax=axs[1], color=colors[i], marker='o', label=name)
axs[1].set_title('Reference spectra')
axs[1].set_xlabel('Wavelength (nm)')
axs[1].set_ylabel(r'$R_{rs}$ (sr$^{-1}$)')
axs[1].legend(fontsize=10);

## Classify the image with the custom spectra

Pass the spectra to {py:class}`GRSl2bgen.owt.OWT` with `xowt` (dimensions `owt` and
`wl`). They are interpolated on the image bands and compared with each pixel spectrum
with the spectral angle (bands within 350--800 nm by default):

$$
\theta_j = \arccos\left(\frac{\mathbf{R}\cdot\mathbf{R}_j}{\lVert\mathbf{R}\rVert\,\lVert\mathbf{R}_j\rVert}\right)
$$

Each pixel gets the class with the smallest angle.

In [ ]:
owt_img = GRSl2bgen.OWT(raster, xowt=xowt_img)
xowt_prod = owt_img.multi_process().where(raster.mask == 0).compute()

def show_owt(ax, prod, labels, cmap, title):
    """Map an OWT index with the class names on the colour bar."""
    n = len(labels)
    im = prod.owt_index.plot.imshow(ax=ax, cmap=cmap, vmin=0.5, vmax=n + 0.5,
                                    cbar_kwargs={'ticks': range(1, n + 1), 'shrink': 0.8, 'label': ''})
    im.colorbar.ax.set_yticklabels(labels)
    show_map(ax, title)

fig, axs = plt.subplots(1, 2, figsize=(17, 6.5), constrained_layout=True)
show_owt(axs[0], xowt_prod, names, cmap_img, 'OWT from image spectra')
xowt_prod.owt_dist.plot.imshow(ax=axs[1], cmap='RdBu_r', vmin=0, vmax=0.4,
                               cbar_kwargs={'label': 'rad', 'shrink': 0.8})
show_map(axs[1], 'Spectral angle to the selected class')

Area covered by each class, $A_j = N_j\,|\Delta x\,\Delta y|$, and mean spectrum of its
pixels (red) compared with the reference spectrum (dotted). A good class has a mean
spectrum close in shape to its reference. The shelf classes cover most of the scene,
while the plume classes follow the turbidity gradient from the estuary mouth. The
"plume core" class gathers pixels less turbid than its reference: the spectral angle
ignores the magnitude, and the shape of the most turbid spectra changes little once
the red and NIR plateau appears.

In [ ]:
pixel_area_km2 = dx * dy * 1e-6
index = xowt_prod.owt_index
Ntot = int(index.count())

ncols = 3
nrows = int(np.ceil(Nowt / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.6 * nrows), constrained_layout=True)
rows = []
for j, (ax, name) in enumerate(zip(axs.ravel(), names), start=1):
    Rrs_j = Rrs.where(index == j)
    Npix = int(Rrs_j.isel(wl=0).count())
    rows.append({'OWT': name, 'pixels': Npix, 'area (km2)': Npix * pixel_area_km2,
                 'share (%)': 100 * Npix / Ntot})
    if Npix:
        Rrs_j.mean(['x', 'y']).plot(ax=ax, color='r', label='mean of class')
    xowt_img.sel(owt=name).plot(ax=ax, color=colors[j - 1], ls=':', marker='o', ms=3, label='reference')
    ax.set_title(f'{name} ({100 * Npix / Ntot:.1f} %)', fontsize=11)
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel(r'$R_{rs}$ (sr$^{-1}$)')
axs.ravel()[0].legend(fontsize=9)
for ax in axs.ravel()[Nowt:]:
    ax.set_axis_off()
pd.DataFrame(rows).set_index('OWT').round(2)

## Compare with the built-in databases

Classify the same area with the Spyrakos et al. (2018) and Bi and Hieronymi (2024)
databases. Besides the class maps, the spectral angle measures how well each database
represents the pixels: the smaller the angle, the better the match.

In [ ]:
owt_spy = GRSl2bgen.OWT(raster, owt_database='Spyrakos2018')
prod_spy = owt_spy.multi_process().where(raster.mask == 0).compute()
owt_bi = GRSl2bgen.OWT(raster, owt_database='Bi2024')
prod_bi = owt_bi.multi_process().where(raster.mask == 0).compute()

fig, axs = plt.subplots(1, 2, figsize=(17, 6.5), constrained_layout=True)
show_owt(axs[0], prod_spy, [str(i) for i in range(1, 14)], owt_spy.cmap_owt, 'Spyrakos et al. (2018)')
show_owt(axs[1], prod_bi, ['1', '2', '3a', '3b', '4a', '4b', '5a', '5b', '6', '7'], owt_bi.cmap_owt,
         'Bi and Hieronymi (2024)')

In [ ]:
angles = {'image spectra': xowt_prod.owt_dist, 'Spyrakos2018': prod_spy.owt_dist, 'Bi2024': prod_bi.owt_dist}
bins = np.linspace(0, 0.6, 61)
fig, ax = plt.subplots(figsize=(9, 4.5))
for label, da in angles.items():
    values = da.values.ravel()
    values = values[np.isfinite(values)]
    ax.hist(values, bins=bins, histtype='step', lw=2, label=f'{label} (median {np.median(values):.3f} rad)')
ax.set_xlabel(r'Spectral angle $\theta$ of the selected class (rad)')
ax.set_ylabel('Number of pixels')
ax.legend();

The image spectra give the smallest angles (see the medians in the legend), followed
by Spyrakos et al. (2018) and by Bi and Hieronymi (2024). Spyrakos et al. (2018) sees
the shelf as a single type (3) and separates the plume in three turbid types (9, 12,
2), whereas the Bi2024 types split the scene in the clear (3a, 3b) and moderately to
highly turbid (4a, 4b, 5a) types. Image spectra fit this scene best, but they are
specific to it and to this date, and they have no link with water constituents
unless you characterise the selected sites.

Even with the image spectra, the large angles along the fronts and filaments north of
the plume (red in the angle map of the previous section) show that a few spectra
cannot represent every water mass of a dynamic plume, nor pixels contaminated by
floating material or ship wakes.

## Build a combined database: Bi2024 + phytoplankton types

External spectra can be merged with a built-in database. Here, the Bi and Hieronymi
(2024) types are complemented by the median spectra of six phytoplankton bloom types
from Gernez et al. (2023) ("50 shades" of phytoplankton blooms: cyanobacteria,
dinoflagellates, *Lepidodinium*, *Mesodinium* and red *Noctiluca*).

```{note}
The Gernez et al. (2023) spectra are not distributed with GRSl2bgen; set `owt_dir` to
your own copy.
```

In [ ]:
owt_dir = '/DATA/data/model/owt'

gernez = []
for file in sorted(glob.glob(os.path.join(owt_dir, 'Gernez_50_shades_Rrs_cluster_*.txt'))):
    name = file.split('_')[-1].replace('.txt', '')
    # one column per in situ spectrum of the cluster
    cluster = pd.read_csv(file, sep=r'\s+').set_index('wavelength').stack().to_xarray()
    gernez.append(cluster.median('level_1').assign_coords(owt=name))
gernez = xr.concat(gernez, 'owt').rename({'wavelength': 'wl'}).reset_coords(drop=True)
print(gernez.owt.values, 'at', gernez.wl.values, 'nm')

The two sources are not in the same unit: the Bi2024 mean spectra are standardised
($nR_{rs}$, nm$^{-1}$) while the Gernez spectra are in sr$^{-1}$. This does not matter
for the classification, since the spectral angle ignores the magnitude of the
spectra, so they can be combined as they are. To compare them visually, every
spectrum is standardised by its integral over the 9 Sentinel-2 bands of the Gernez
spectra, $nR_{rs}(\lambda) = R_{rs}(\lambda) / \int R_{rs}\,\mathrm{d}\lambda$.

In [ ]:
def standardise(spectra):
    return spectra / spectra.integrate('wl')

bi = owt_bi.owt.interp(wl=gernez.wl)          # Bi2024 standardised mean spectra (m_nRrs)
bi['owt'] = ['1', '2', '3a', '3b', '4a', '4b', '5a', '5b', '6', '7']
xowt_comb = xr.concat([bi, gernez], dim='owt').dropna('wl')
names_comb = xowt_comb.owt.values
Ncomb = len(names_comb)
cmap_comb = mpl.colors.ListedColormap(plt.get_cmap('tab20')(np.arange(Ncomb)))

fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(names_comb):
    standardise(xowt_comb).sel(owt=name).plot(ax=ax, color=cmap_comb(i), marker='o', ms=3,
                                 ls='-' if i < 10 else '--', label=name)
ax.set_title('Bi2024 (solid) + Gernez et al. 2023 (dashed), standardised')
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel(r'$nR_{rs}$ (nm$^{-1}$)')
ax.legend(ncol=2, fontsize=9, bbox_to_anchor=(1.01, 1), loc='upper left');

This combined database is the **Tarasenko et al. (2025)** database shipped with
GRSl2bgen (`owt_index_Ta2025` in the L2B product). The packaged file stores the same
spectra, with the phytoplankton types in another order; the table gives the class
numbers used in the L2B product:

In [ ]:
ta2025 = xr.open_dataarray(GRSl2bgen.owt.OWT_tarasenko2025_file)
print('same classes:', sorted(ta2025.owt.values) == sorted(names_comb))
print('same spectra:', np.allclose(ta2025.transpose('owt', 'wl').values,
                                   xowt_comb.sel(owt=ta2025.owt.values).values, rtol=1e-6))
# class numbers used in owt_index_Ta2025
pd.Series(ta2025.owt.values, index=range(1, len(ta2025.owt) + 1), name='Ta2025 class').to_frame().T

### Separability of the classes

Two classes with similar shapes cannot be told apart by the SAM. The spectral angle
between every pair of reference spectra shows which classes risk being confused
(small angles, light colours):

In [ ]:
unit = xowt_comb / np.sqrt((xowt_comb ** 2).sum('wl'))
cosines = np.clip(unit.values @ unit.values.T, -1, 1)
angle_matrix = pd.DataFrame(np.degrees(np.arccos(cosines)), index=names_comb, columns=names_comb)
np.fill_diagonal(angle_matrix.values, np.nan)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(angle_matrix, cmap='magma', vmin=0, vmax=40)
ax.set_xticks(range(Ncomb), names_comb, rotation=60, ha='right')
ax.set_yticks(range(Ncomb), names_comb)
for i in range(Ncomb):
    for j in range(Ncomb):
        if i != j:
            ax.text(j, i, f'{angle_matrix.iat[i, j]:.0f}', ha='center', va='center', fontsize=8,
                    color='white' if angle_matrix.iat[i, j] < 20 else 'black')
fig.colorbar(im, ax=ax, label='Spectral angle (degrees)')
ax.set_title('Spectral angle between reference spectra');

Bi2024 types 3a and 3b (3°), and 4a and 4b (2°), have almost the same shape on the
Sentinel-2 bands: in Bi and Hieronymi (2024) the "b" types mainly differ by their
higher reflectance (strongly scattering blooms), which the spectral angle ignores.
They are therefore hardly separable with the SAM; the Euclidean distance
(`spectral_distance='euclidean'`, with spectra in sr$^{-1}$) is sensitive to this
difference in magnitude. The phytoplankton types are well separated from the
clear-water types, but closer to each other and to types 5b, 6 and 7.

Classify the image with the combined database:

In [ ]:
owt_comb = GRSl2bgen.OWT(raster, xowt=xowt_comb)
prod_comb = owt_comb.multi_process().where(raster.mask == 0).compute()

fig, axs = plt.subplots(1, 2, figsize=(17, 6.5), constrained_layout=True)
show_owt(axs[0], prod_comb, names_comb, cmap_comb, 'Bi2024 + Gernez et al. (2023)')
prod_comb.owt_dist.plot.imshow(ax=axs[1], cmap='RdBu_r', vmin=0, vmax=0.4,
                               cbar_kwargs={'label': 'rad', 'shrink': 0.8})
show_map(axs[1], 'Spectral angle')

No pixel is assigned to a phytoplankton bloom type: the scene is dominated by
sediment-driven types (3b on the shelf, 4a, 4b and 5a in the plume). The angles are
much larger than with the image spectra, in particular in the north-west, which
none of the generic types represents well.

## Simulated spectra of harmful algal species

Reference spectra can also come from radiative transfer simulations. The HABLAB file
below contains $R_{rs}$ simulated for 14 phytoplankton species (identified by a
3-letter code) at two chlorophyll-a concentrations (20 and 200 mg m$^{-3}$), with
(`Rrs_tot`) and without (`Rrs_elas`) the inelastic contributions, as named in the file.

```{note}
This file is not distributed with GRSl2bgen.
```

In [ ]:
hablab = xr.open_dataset(os.path.join(owt_dir, 'hablab_owt_chl20_chl_200.nc'))
species = hablab.species.values
cmap_sp = plt.get_cmap('Spectral_r')

fig, axs = plt.subplots(1, 2, figsize=(15, 5), sharey=False, constrained_layout=True)
for ax, chl in zip(axs, hablab.chl.values):
    for i, sp in enumerate(species):
        color = cmap_sp(i / (len(species) - 1))
        sel = hablab.sel(chl=chl, species=sp)
        sel.Rrs_elas.plot(ax=ax, color=color, ls=':')
        sel.Rrs_tot.plot(ax=ax, color=color, label=sp)
    ax.set_title(f'Chl-a = {chl} mg m$^{{-3}}$ (dotted: Rrs_elas)')
    ax.set_xlabel('Wavelength (nm)')
    ax.set_ylabel(r'$R_{rs}$ (sr$^{-1}$)')
    ax.set_xlim(400, 800)
axs[1].legend(ncol=2, fontsize=9);

Combine the species spectra at 200 mg m$^{-3}$ with the Bi2024 types and classify the
image. Classes are standardised as above.

In [ ]:
sp_200 = hablab.Rrs_tot.sel(chl=200).rename({'species': 'owt'}).reset_coords(drop=True)
xowt_hab = standardise(xr.concat([sp_200.interp(wl=gernez.wl), bi], dim='owt').dropna('wl'))
names_hab = xowt_hab.owt.values
cmap_hab = mpl.colors.ListedColormap(plt.get_cmap('tab20', len(names_hab))(np.arange(len(names_hab))))

owt_hab = GRSl2bgen.OWT(raster, xowt=xowt_hab)
prod_hab = owt_hab.multi_process().where(raster.mask == 0).compute()

fig, axs = plt.subplots(1, 2, figsize=(17, 6.5), constrained_layout=True)
show_owt(axs[0], prod_hab, names_hab, cmap_hab, 'HABLAB species (200 mg m$^{-3}$) + Bi2024')
prod_hab.owt_dist.plot.imshow(ax=axs[1], cmap='RdBu_r', vmin=0, vmax=0.4,
                              cbar_kwargs={'label': 'rad', 'shrink': 0.8})
show_map(axs[1], 'Spectral angle')

## Advanced settings

### Restrict the spectral range

`wl_range` selects the bands used in the SAM (default 350--800 nm). Restricting it to
the green to red-edge range (550--740 nm) focuses on the phytoplankton absorption and
red-edge features and removes the blue bands, which are more affected by atmospheric
correction uncertainties. Here the target is a single species (*Mesodinium*, `MES`)
against the Bi2024 types:

In [ ]:
xowt_mes = standardise(xr.concat([sp_200.sel(owt=['MES']).interp(wl=gernez.wl), bi], dim='owt').dropna('wl'))
names_mes = xowt_mes.owt.values
cmap_mes = mpl.colors.ListedColormap(plt.get_cmap('tab20', len(names_mes))(np.arange(len(names_mes))))

fig, axs = plt.subplots(1, 3, figsize=(22, 6.5), constrained_layout=True)
settings = [('350-800 nm (default)', raster, {}),
            ('550-740 nm', raster, {'wl_range': slice(550, 740)}),
            ('bands 490, 560, 665 nm', raster.sel(wl=[490, 560, 665]), {})]
for ax, (title, image, kwargs) in zip(axs, settings):
    prod = GRSl2bgen.OWT(image, xowt=xowt_mes, **kwargs).multi_process().where(raster.mask == 0).compute()
    show_owt(ax, prod, names_mes, cmap_mes, title)

### Select specific bands

As in the third map above, select bands of the image before the classification
(`raster.sel(wl=[...])`) to use only those bands: for example, to mimic a sensor with
fewer bands or to exclude a band with known issues.

The three maps differ substantially: the bands used define which spectral features
can separate the classes. With the 550--740 nm range, the shelf switches from type 3b
to type 3a, two types with almost the same shape. With only three visible bands, the
plume core is assigned to `MES`: its high red reflectance mimics the red peak of a
dense *Mesodinium* bloom once the red-edge and NIR bands, where sediments and
phytoplankton differ, are removed. Such a result must not be read as a bloom
detection.

### Interactive exploration

In Jupyter, explore the result interactively with
[hvplot](https://hvplot.holoviz.org/) (not rendered in this documentation):

    import hvplot.xarray
    xowt_prod.hvplot.explorer(cmap='RdYlBu_r', height=600, width=700)

## Summary

- A multiscale Zarr L2A store is opened by group (`xr.open_zarr(path, group='0')`)
  and passed as a dataset to `GRSl2bgen.Product`.
- Any set of spectra with dimensions `owt` and `wl` can be used as OWT reference with
  `GRSl2bgen.OWT(raster, xowt=...)`.
- Image-derived spectra fit the scene best and resolve the sediment gradient of the
  plume, but they are specific to this scene and date; built-in databases are
  generic and coarser over turbid coastal waters.
- Standardise spectra from different sources before combining them, check the
  angles between classes to make sure they are separable, and keep the red-edge and
  NIR bands to separate sediment-rich waters from phytoplankton blooms.

## References

- Bi, S., Hieronymi, M. (2024). Holistic optical water type classification for ocean,
  coastal, and inland waters. *Limnology and Oceanography*.
- Gernez, P., et al. (2023). The many shades of red tides: Sentinel-2 optical types of
  highly-concentrated harmful algal blooms. *Remote Sensing of Environment*, 287, 113486.
- Spyrakos, E., et al. (2018). Optical types of inland and coastal waters.
  *Limnology and Oceanography*, 63, 846--870.